# Questão 3 - Mapa do Brasil

Crie um sistema para representar o mapa do Brasil através dos seus estados.

Esse sistema, além de representar o país, também deverá exibir os vizinhos diretos de cada estado.

### Exemplo

O estado de **SP** possui como vizinhos diretos:

- PR
- MS
- MG
- RJ

### O que deve ser feito

Permita que o sistema receba dois parâmetros:

- um estado de origem;
- um estado de destino.

A partir desses parâmetros, o sistema deverá retornar um caminho possível para ir do estado de origem até o estado de destino.


## Resolução

Para resolver este problema utilizaremos a estrutura de dados **Grafo**.

O mapa do Brasil e suas fronteiras se encaixam perfeitamente na definição de um grafo não direcionado.
- Vértices (Nós): Representam os estados (ex: SP, PR, MS).
- Arestas (Conexões): Representam as fronteiras físicas entre os estados (ex: a fronteira que liga SP ao PR).

#### 1. Grafo
Encapsula um Dicionário que funciona como uma **Lista de Adjacências**. O dicionário armazena qual estado se liga a quais. A classe tem a inteligência de garantir que, ao `adicionar_aresta(origem, destino)`, a ligação seja **bidirecional** (se SP faz fronteira com MG, automaticamente MG faz fronteira com SP).


In [1]:
class Grafo:
    def __init__(self):
        self.vertices = {}
        
    def adicionar_vertice(self, estado):
        if estado not in self.vertices:
            self.vertices[estado] = []
            
    def adicionar_aresta(self, origem, destino):
        # Garante que os vértices existem
        self.adicionar_vertice(origem)
        self.adicionar_vertice(destino)
        
        # Adiciona a conexão bidirecional (grafo não direcionado)
        if destino not in self.vertices[origem]:
            self.vertices[origem].append(destino)
        if origem not in self.vertices[destino]:
            self.vertices[destino].append(origem)
            
    def obter_vizinhos(self, estado):
        if estado in self.vertices:
            return self.vertices[estado]
        return []

    def existe_vertice(self, estado):
        return estado in self.vertices


#### 2. Fila
Implementa a lógica FIFO (First-In, First-Out / O primeiro que entra é o primeiro que sai). Essa fila será utilizada para controlar a ordem em que os estados são visitados.

In [2]:
class Fila:
    def __init__(self):
        self.itens = []
        
    def enfileirar(self, item):
        self.itens.append(item)
        
    def desenfileirar(self):
        if not self.esta_vazia():
            return self.itens.pop(0)
        return None
        
    def esta_vazia(self):
        return len(self.itens) == 0


#### 3. Algoritmo de Busca (BFS)
Para descobrir a rota entre o Estado A e o Estado B, utilizamos a função `encontrar_caminho`, que aplica o algoritmo de **Busca em Largura (Breadth-First Search)**.

1. Começamos no estado de origem. Ele é marcado como **visitado** (para não andarmos em círculos depois) e inserido na `Fila` carregando o caminho até ele (que no começo é apenas [origem]).
2/ O algoritmo tira o primeiro da fila e usa a classe `Grafo` para saber quem são seus **vizinhos**.
3. Se um dos vizinhos for o nosso **destino final**, o trabalho acabou: ele devolve o caminho formatado.
4. Se o vizinho não for o destino (e ainda não tiver sido visitado antes), ele é colocado de volta no final da `Fila`, agora carregando o histórico da viagem até chegar nele.
5. O ciclo repete até o destino ser achado.

- **Por que a Busca em Largura (BFS)?** Como ela expande visitando todos os vizinhos da distância 1, e depois todos da distância 2, o primeiro trajeto que bater no estado de destino será matematicamente o trajeto mais curto (o que precisa cruzar a menor quantidade de fronteiras).

In [3]:
# 3. Algoritmo de Busca em Largura (BFS) utilizando a classe Grafo
def encontrar_caminho(origem, destino, grafo):
    origem = origem.upper()
    destino = destino.upper()
    
    if not grafo.existe_vertice(origem) or not grafo.existe_vertice(destino):
        return "Estado(s) inválido(s)."
    
    if origem == destino:
        return [origem]
    
    # Utilizando a estrutura Fila
    fila = Fila()
    fila.enfileirar((origem, [origem]))
    
    visitados = set([origem])
    
    while not fila.esta_vazia():
        estado_atual, caminho = fila.desenfileirar()
        
        for vizinho in grafo.obter_vizinhos(estado_atual):
            if vizinho == destino:
                return caminho + [vizinho]
            
            if vizinho not in visitados:
                visitados.add(vizinho)
                fila.enfileirar((vizinho, caminho + [vizinho]))
                
    return "Caminho não encontrado."


#### 4. Popular o Grafo com o Mapa do Brasil
Instanciamos o `mapa_brasil` (um objeto do tipo `Grafo`). Através de um mapa de conexões conhecido (todas as fronteiras brasileiras reais), um loop utiliza a função adicionar_aresta para desenhar todo o país na memória do computador.

In [4]:
# 2. Popular o Grafo com o Mapa do Brasil
mapa_brasil = Grafo()

conexoes = {
    'AC': ['AM', 'RO'],
    'AL': ['PE', 'SE', 'BA'],
    'AP': ['PA'],
    'AM': ['RR', 'PA', 'MT', 'RO', 'AC'],
    'BA': ['SE', 'AL', 'PE', 'PI', 'TO', 'GO', 'MG', 'ES'],
    'CE': ['PI', 'PE', 'PB', 'RN'],
    'DF': ['GO', 'MG'],
    'ES': ['BA', 'MG', 'RJ'],
    'GO': ['TO', 'BA', 'MG', 'DF', 'MS', 'MT'],
    'MA': ['PA', 'TO', 'PI'],
    'MT': ['AM', 'PA', 'TO', 'GO', 'MS', 'RO'],
    'MS': ['MT', 'GO', 'MG', 'SP', 'PR'],
    'MG': ['BA', 'ES', 'RJ', 'SP', 'MS', 'GO', 'DF'],
    'PA': ['AP', 'MA', 'TO', 'MT', 'AM', 'RR'],
    'PB': ['RN', 'CE', 'PE'],
    'PR': ['SP', 'MS', 'SC'],
    'PE': ['PB', 'CE', 'PI', 'BA', 'AL'],
    'PI': ['MA', 'TO', 'BA', 'PE', 'CE'],
    'RJ': ['ES', 'MG', 'SP'],
    'RN': ['CE', 'PB'],
    'RS': ['SC'],
    'RO': ['AM', 'MT', 'AC'],
    'RR': ['AM', 'PA'],
    'SC': ['PR', 'RS'],
    'SP': ['MG', 'RJ', 'PR', 'MS'],
    'SE': ['AL', 'BA'],
    'TO': ['PA', 'MA', 'PI', 'BA', 'GO', 'MT']
}

# Registrando cada estado e suas arestas (fronteiras) no objeto Grafo
for estado, vizinhos in conexoes.items():
    for vizinho in vizinhos:
        mapa_brasil.adicionar_aresta(estado, vizinho)


#### 4. Testar o algoritmo

In [5]:
print("De SP para BA:", encontrar_caminho('SP', 'BA', mapa_brasil))
print("De RS para CE:", encontrar_caminho('RS', 'CE', mapa_brasil))
print("De AC para ES:", encontrar_caminho('AC', 'ES', mapa_brasil))


De SP para BA: ['SP', 'MG', 'BA']
De RS para CE: ['RS', 'SC', 'PR', 'MS', 'GO', 'BA', 'PE', 'CE']
De AC para ES: ['AC', 'AM', 'PA', 'TO', 'BA', 'ES']


## Complexidade de algoritmos

Análise assintótica (Notação Big-O) de Tempo para o algoritmo implmentado:

Considerando **V** como o número de vértices (estados, que são 27) e **E** como o número de arestas (fronteiras entre os estados).


A complexidade de tempo clássica da Busca em Largura (BFS) é **O(V + E)**. No entanto, analisando a sua implementação detalhadamente, temos as seguintes operações:

1. **Visita aos vértices e arestas**: O loop `while` processa cada vértice da fila no máximo uma vez, e o `for` itera sobre os vizinhos de cada vértice. Juntos, eles garantem que todos os vértices e arestas do grafo sejam acessados uma vez (ou duas, por ser bidirecional). Isso nos dá a base de **O(V + E)**.

2. **Busca no Set (visitados)**: A verificação `if vizinho not in visitados`: custa **O(1)** em média, pois visitados é implementado com um *Set* (tabela hash).

3. **Cópia do Caminho**: A linha `fila.enfileirar((vizinho, caminho + [vizinho]))` cria uma nova lista para salvar o histórico a cada iteração. No pior dos cenários (se o mapa fosse uma linha reta), o caminho poderia ter tamanho **V**. Como isso é feito para cada vértice inserido na fila, adiciona um custo na construção dos caminhos, elevando a complexidade teórica para **O(V² + E)**.


Desta forma, a complexidade é **O(V² + E)** devido ao custo de se criar as listas com o histórico dos caminhos na fila. Entretanto, como o número de estados do Brasil é fixo e muito pequeno **(V = 27)**, o tempo de execução real do algoritmo será instantâneo, equivalente a um tempo constante **O(1)** em termos práticos.


Referência Bibliográfica:
- Entendendo Estruturas de Dados por Marcello La Rocca.
- Python Official Wiki - Time Complexity (https://wiki.python.org/moin/TimeComplexity)